<a href="https://colab.research.google.com/github/ibnu-aulia/arabic-text-analyzer/blob/main/arabic_sentiment_first_neural_net.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Arabic Sentiment: My First Neural Network

Goal: classify short Arabic sentences as good (1) or bad (0) reviews.

How it works: code counts good and bad words in each sentence, then a
2-layer PyTorch network is trained on those two numbers.

In [1]:
import torch
import torch.nn as nn

# 8 short Arabic reviews
sentences = [
    "الفيلم جميل جدا",
    "الطعام سيء جدا",
    "المطعم رائع جدا",
    "الكتاب ممل جدا",
    "الفيلم ممتاز جدا",
    "الطعام رديء جدا",
    "الفندق رائع جدا",
    "الصوت سيء جدا",
]

# 1 = good review, 0 = bad review
labels = torch.tensor([[1.0], [0.0], [1.0], [0.0], [1.0], [0.0], [1.0], [0.0]])

good_words = ["جميل", "ممتاز", "رائع"]
bad_words = ["سيء", "رديء", "ممل"]

# turn one sentence into [good count, bad count]
def count_words(sentence):
    good_count = 0.0
    bad_count = 0.0
    for word in sentence.split():
        if word in good_words:
            good_count += 1
        if word in bad_words:
            bad_count += 1
    return [good_count, bad_count]

# build one row of features per sentence
rows = []
for sentence in sentences:
    rows.append(count_words(sentence))

features = torch.tensor(rows)
print(features)
print(features.shape)

tensor([[1., 0.],
        [0., 1.],
        [1., 0.],
        [0., 1.],
        [1., 0.],
        [0., 1.],
        [1., 0.],
        [0., 1.]])
torch.Size([8, 2])


In [2]:
# use the GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

# move data to the device
features = features.to(device)
labels = labels.to(device)

# network: 2 inputs -> 3 neurons -> ReLU -> 1 neuron -> sigmoid
model = nn.Sequential(
    nn.Linear(2, 3),
    nn.ReLU(),
    nn.Linear(3, 1),
    nn.Sigmoid()
).to(device)

loss_fn = nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

# training loop: forward, loss, backward, update
for step in range(500):
    optimizer.zero_grad()
    out = model(features)
    loss = loss_fn(out, labels)
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(step, loss.item())

cuda
0 0.7243200540542603
100 0.42759373784065247
200 0.1342901885509491
300 0.07068250328302383
400 0.046493370085954666


In [3]:
# accuracy on the training sentences (no gradients needed)
with torch.no_grad():
    out = model(features)
    predictions = (out > 0.5).float()
    correct = (predictions == labels).float()
    accuracy = correct.mean()

print(accuracy.item())

1.0


## Conclusion

The code counts good and bad words in each Arabic sentence, turning
every sentence into two numbers. A 2-layer PyTorch network was then
trained on these numbers to predict whether a review is good or bad.

After 500 epochs on a T4 GPU, the loss dropped to about 0.05 and the
accuracy reached 1.0.

## Limitations

- Accuracy was measured on the same 8 sentences used for training,
  so it does not show how the model handles new sentences.
- Words are matched exactly. A word with the definite article
  (e.g. السيء instead of سيء) or a spelling variant (hamza/alif forms,
  ي vs ى) is not counted, so such a sentence can look neutral.

## Next steps

- Normalize Arabic text before counting (strip ال, unify alif and ya forms).
- Test on sentences the model has not seen during training.